# Stage 3 — Sentiment Analysis (NLP Models)

Fine-tunes every model in `NLP_MODELS` on the weak-labeled AMZN headlines
from Stage 1, repeated across every seed in `CONFIG["random_seeds"]`. The
full sweep is `len(NLP_MODELS) × len(random_seeds)` runs (currently 4 × 10 = 40),
which is enough to compute mean ± std and run statistical significance tests
in Stage 6.

## What this stage produces
- `./results/nlp_results.csv` — one row per (model, seed) with accuracy, F1,
  train/inference times, and ISO 8601 UTC start/end timestamps for both phases.
- `./artifacts/nlp_probs.parquet` — per-headline 3-class probabilities from the
  single best (model, seed) combination, used by Stage 4 to build rolling
  sentiment features for the regression `+NLP` variant.
- `./artifacts/nlp_probs_run_meta.json` — sidecar metadata describing which
  run produced the probability file and its wall-clock timing windows.

## What this stage consumes
- `./artifacts/text_df.parquet` (weak-labeled headlines, from Stage 1).
- `./artifacts/num_df.parquet` (only used to compute the shared cutoff date).

## Energy join hook
Training and inference are bracketed with both `time.time()` (for fast
per-run summaries) and `datetime.now(timezone.utc).isoformat()` (for joining
against an external power log post-run). Energy is **not** measured by Python
here — a separate wall-meter / RAPL log is integrated over each run's
`[wall_*_start_iso, wall_*_end_iso]` window after the pipeline finishes.

## Best-model selection
The "best" run is picked by **F1, not accuracy**, because the weak labels are
class-imbalanced (mostly neutral) and accuracy can be inflated by always
predicting the majority class. F1 (weighted) penalizes that failure mode.


In [1]:
# Load shared helpers, config values, and model registries.
from common import *
from datetime import datetime, timezone

# Hugging Face training utilities used in this stage.
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2ForSequenceClassification,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
)

# text_df contains weak-labeled AMZN headlines from Stage 1.
text_df = load_text_df()
# num_df is loaded only to compute a shared split cutoff date.
num_df = load_num_df()

# Normalize timestamps to date-level precision for clean chronological splits.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# cutoff_date defines the train/test boundary used by both text and numeric tasks.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)
print(f"Chronological cutoff: {cutoff_date.date()}")

# Collector object accumulates per-run metrics for CSV export.
results = ResultsCollector()

/home/hrilab/energy-analysis-pipeline/lab_machine_pkg/.venv/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


[common] device=cuda  artifacts=/home/hrilab/energy-analysis-pipeline/lab_machine_pkg/artifacts  results=/home/hrilab/energy-analysis-pipeline/lab_machine_pkg/results


Chronological cutoff: 2023-09-25


## 3.1 Training routine

In [2]:
def build_tokenizer_and_model(model_path, config):
    """Create tokenizer/model pair for a checkpoint path."""
    # GPT-2 special-case: the GPT-2 family is currently commented out of NLP_MODELS,
    # but this branch is preserved so that re-enabling GPT-2 is a one-line registry
    # change (just uncomment the entries in common.py). GPT-2 needs a pad token
    # (its tokenizer ships without one — we reuse EOS) and the corresponding
    # pad_token_id must be set on model.config so attention masks work correctly.
    # Treat this branch as dead-by-design, not dead-by-accident.
    is_gpt2 = "gpt2" in model_path
    if is_gpt2:
        tokenizer = GPT2Tokenizer.from_pretrained(model_path)
        tokenizer.pad_token = tokenizer.eos_token

        model = GPT2ForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        # Non-GPT models can use Auto classes directly.
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )

    # Move model to GPU/CPU selected in common.py.
    model.to(DEVICE)
    return tokenizer, model


def train_nlp_model(model_name, model_path, text_df, config, seed, cutoff_date):
    """Train one model for one seed and return metrics + predictions."""
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=config)

    # Build chronological train/test datasets to avoid temporal leakage.
    train_ds, test_ds, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=config["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    # run_dir stores temporary trainer outputs for this run.
    run_dir = RESULTS_DIR / f"{model_name.replace(' ', '_')}_seed{seed}"
    # eval_strategy="epoch" runs the test-set evaluation at the end of each
    # training epoch, which gives us a learning curve and lets us catch
    # divergence early. Side-effect worth disclosing: the value reported in
    # `train_time_s` includes those per-epoch evaluations, not just gradient
    # updates. The energy-join numbers (wall_train_start_iso/end_iso) bracket
    # the same span, so external energy is consistent with reported time.
    training_args = TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["train_batch_size"],
        per_device_eval_batch_size=config["eval_batch_size"],
        learning_rate=config["learning_rate"],
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        seed=seed,
        report_to="none",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=config.get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_clf_metrics,
    )

    # ISO timestamps are written so that an external power log (sampled by a
    # wall meter or RAPL outside this notebook) can be integrated over each
    # [start, end] window after the run completes. They are NOT used for
    # internal timing — `time.time()` deltas handle that. Both methods bracket
    # the same code so the windows agree to within a millisecond.
    wall_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    wall_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Evaluate on chronological test split.
    eval_out = trainer.evaluate()

    # Measure prediction/inference time + wall-clock anchors for external energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    pred_out = trainer.predict(test_ds)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # Convert logits to class IDs via argmax.
    preds = np.argmax(pred_out.predictions, axis=-1)

    # record stores metrics that will be appended to nlp_results.csv.
    record = {
        "model": model_name,
        "seed": seed,
        "accuracy": eval_out["eval_accuracy"],
        "f1": eval_out["eval_f1"],
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(test_ds),
        "wall_train_start_iso": wall_train_start_iso,
        "wall_train_end_iso": wall_train_end_iso,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }

    # Free memory before the next model/seed run.
    del model, trainer
    torch.cuda.empty_cache()

    return record, preds


def train_best_model_and_export_probs(model_name, model_path, seed, text_df, cutoff_date):
    """Retrain best run and export per-headline class probabilities."""
    # We retrain the best (model, seed) from scratch instead of reusing a model
    # already in memory because the main sweep above issues `del model, trainer`
    # at the end of every run to free GPU memory between configurations. By the
    # time we know which run "won" (post-sweep), the winning weights are gone.
    #
    # Cost: ~10% extra compute relative to the main sweep (one extra training
    # of the best configuration). This is flagged as a future optimization;
    # the simplest fix is to checkpoint each trained model to disk during the
    # sweep and reload the winner here.
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=CONFIG["nlp"])

    # Fit using chronological training slice only.
    train_ds, _, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    output_dir = RESULTS_DIR / f"best_{model_name.replace(' ', '_')}_seed{seed}"
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=CONFIG["nlp"]["epochs"],
        per_device_train_batch_size=CONFIG["nlp"]["train_batch_size"],
        per_device_eval_batch_size=CONFIG["nlp"]["eval_batch_size"],
        learning_rate=CONFIG["nlp"]["learning_rate"],
        save_strategy="no",
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=CONFIG["nlp"].get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
    )
    wall_best_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    trainer.train()
    best_train_time = time.time() - t0
    wall_best_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Score EVERY headline (train slice + test slice), not just test. Stage 4's
    # `+NLP` variant builds rolling-K-day sentiment features by averaging
    # per-headline probabilities over a lookback window, which means it needs a
    # probability for every headline date in the analysis window — including
    # train-period headlines that contribute to the rolling average for the
    # earliest test-period rows. Labels are placeholders (zeros) because we
    # only consume the model's softmax output here, never compare to truth.
    infer_ds = SentimentDataset(
        texts=text_df["text"],
        labels=pd.Series(np.zeros(len(text_df), dtype=int)),
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
    )

    wall_best_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    prediction_output = trainer.predict(infer_ds)
    best_infer_time = time.time() - t0
    wall_best_infer_end_iso = datetime.now(timezone.utc).isoformat()
    logits = prediction_output.predictions
    # Softmax converts logits into class probabilities summing to 1.
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    # Save one row per headline so Stage 4 can build rolling sentiment features.
    nlp_probs_df = pd.DataFrame(
        {
            "date": text_df["date"].values,
            "prob_negative": probs[:, LABEL_TO_ID["negative"]],
            "prob_neutral": probs[:, LABEL_TO_ID["neutral"]],
            "prob_positive": probs[:, LABEL_TO_ID["positive"]],
            "model": model_name,
            "seed": seed,
        }
    )

    # Sidecar metadata file. The energy-join script reads this to know which
    # (model, seed) produced nlp_probs.parquet and over what wall-clock windows
    # the best-model retrain + scoring happened, so its energy can be attributed
    # separately from the main 40-run sweep recorded in nlp_results.csv.
    import json
    best_run_meta = {
        "model": model_name,
        "seed": seed,
        "wall_best_train_start_iso": wall_best_train_start_iso,
        "wall_best_train_end_iso": wall_best_train_end_iso,
        "best_train_time_s": best_train_time,
        "wall_best_infer_start_iso": wall_best_infer_start_iso,
        "wall_best_infer_end_iso": wall_best_infer_end_iso,
        "best_infer_time_s": best_infer_time,
        "n_headlines_scored": int(len(text_df)),
    }
    with open(ARTIFACTS_DIR / "nlp_probs_run_meta.json", "w") as f:
        json.dump(best_run_meta, f, indent=2)

    # Free memory after export.
    del model, trainer
    torch.cuda.empty_cache()

    return nlp_probs_df

## 3.2 Run every (model, seed) combination

In [3]:
# nlp_predictions: built but never persisted and never read downstream. Kept for
# optional in-notebook diagnostics (e.g., confusion matrices, error analysis on
# specific seeds). Safe to remove if unused; kept here so a future reader can
# inspect raw predictions without re-running the sweep.
nlp_predictions = {}

# Train every model across every configured seed.
for model_name, model_path in NLP_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        print(f"\n{'=' * 60}")
        print(f"  {model_name}  |  seed {seed}")
        print(f"{'=' * 60}")

        # Train and evaluate one (model, seed) run.
        record, preds = train_nlp_model(
            model_name=model_name,
            model_path=model_path,
            text_df=text_df,
            config=CONFIG["nlp"],
            seed=seed,
            cutoff_date=cutoff_date,
        )

        # Save metrics and predictions for this run.
        results.add_nlp(record)
        nlp_predictions[(model_name, seed)] = preds

        # Print compact run summary.
        print(f"  Accuracy: {record['accuracy']:.4f}  |  F1: {record['f1']:.4f}")
        print(
            f"  Train: {record['train_time_s']:.1f}s  |  "
            f"Infer: {record['infer_time_s']:.1f}s"
        )

# Preview collected NLP metrics.
results.nlp_df().round(6)


  DistilBERT  |  seed 0


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.090000,1.069597,0.457326,0.288886
2,1.064400,1.055538,0.433855,0.375646
3,0.996000,1.062345,0.421764,0.395370


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Accuracy: 0.4218  |  F1: 0.3954
  Train: 11.9s  |  Infer: 0.3s

  DistilBERT  |  seed 1


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.092700,1.058050,0.455903,0.285524
2,1.053300,1.085076,0.424609,0.341556
3,0.955300,1.145800,0.376956,0.370556


  Accuracy: 0.3770  |  F1: 0.3706
  Train: 21.5s  |  Infer: 0.6s

  FinBERT  |  seed 9


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.097700,1.055113,0.455903,0.285524
2,1.067200,1.078231,0.421053,0.355945
3,1.038800,1.091491,0.396871,0.396936


  Accuracy: 0.3969  |  F1: 0.3969
  Train: 21.6s  |  Infer: 0.6s


,model,seed,accuracy,f1,train_time_s,infer_time_s,n_test_samples,wall_train_start_iso,wall_train_end_iso,wall_infer_start_iso,wall_infer_end_iso
0,DistilBERT,0,0.421764,0.395370,11.900027,0.286448,1406,2026-05-06T21:15:43.002829+00:00,2026-05-06T21:15:54.902866+00:00,2026-05-06T21:15:55.172823+00:00,2026-05-06T21:15:55.459276+00:00
1,DistilBERT,1,0.423186,0.396689,11.682194,0.287538,1406,2026-05-06T21:15:55.979571+00:00,2026-05-06T21:16:07.661779+00:00,2026-05-06T21:16:07.932371+00:00,2026-05-06T21:16:08.219915+00:00
2,DistilBERT,2,0.430299,0.411337,11.731285,0.292435,1406,2026-05-06T21:16:08.769363+00:00,2026-05-06T21:16:20.500671+00:00,2026-05-06T21:16:20.776104+00:00,2026-05-06T21:16:21.068546+00:00
3,DistilBERT,3,0.427454,0.402569,11.747625,0.297080,1406,2026-05-06T21:16:21.593864+00:00,2026-05-06T21:16:33.341505+00:00,2026-05-06T21:16:33.620706+00:00,2026-05-06T21:16:33.917792+00:00
4,DistilBERT,4,0.439545,0.400351,11.770108,0.304283,1406,2026-05-06T21:16:34.481870+00:00,2026-05-06T21:16:46.252009+00:00,2026-05-06T21:16:46.536848+00:00,2026-05-06T21:16:46.841136+00:00
5,DistilBERT,5,0.421764,0.397456,11.811939,0.301754,1406,2026-05-06T21:16:47.361848+00:00,2026-05-06T21:16:59.173808+00:00,2026-05-06T21:16:59.454585+00:00,2026-05-06T21:16:59.756346+00:00
6,DistilBERT,6,0.452347,0.421478,11.787628,0.308043,1406,2026-05-06T21:17:00.294647+00:00,2026-05-06T21:17:12.082310+00:00,2026-05-06T21:17:12.367032+00:00,2026-05-06T21:17:12.675081+00:00
7,DistilBERT,7,0.424609,0.401898,11.820570,0.306034,1406,2026-05-06T21:17:13.189122+00:00,2026-05-06T21:17:25.009728+00:00,2026-05-06T21:17:25.298740+00:00,2026-05-06T21:17:25.604781+00:00
8,DistilBERT,8,0.411095,0.399445,11.801558,0.304606,1406,2026-05-06T21:17:26.157419+00:00,2026-05-06T21:17:37.958992+00:00,2026-05-06T21:17:38.254035+00:00,2026-05-06T21:17:38.558647+00:00
9,DistilBERT,9,0.432432,0.412344,11.830230,0.308674,1406,2026-05-06T21:17:39.082635+00:00,2026-05-06T21:17:50.912897+00:00,2026-05-06T21:17:51.209312+00:00,2026-05-06T21:17:51.517992+00:00


## 3.3 Persist results and export best-model headline probabilities

In [4]:
# Save Stage 3 metrics to results/nlp_results.csv.
results.save(RESULTS_DIR)

# Identify the best run by F1 so Stage 4 can consume a single probability file.
nlp_results_df = results.nlp_df().copy()
best_row = nlp_results_df.sort_values("f1", ascending=False).iloc[0]

# Extract model ID and seed of the top run.
best_model_name = best_row["model"]
best_seed = int(best_row["seed"])
best_model_path = NLP_MODELS[best_model_name]

print("Best NLP run used for probability export:")
print(best_row.to_string())

# Re-train that best configuration and export per-headline probabilities.
nlp_probs_df = train_best_model_and_export_probs(
    model_name=best_model_name,
    model_path=best_model_path,
    seed=best_seed,
    text_df=text_df,
    cutoff_date=cutoff_date,
)
save_nlp_probs_df(nlp_probs_df)

print(f"Total NLP experiments: {len(results.nlp_results)}")
print(f"Saved per-headline probabilities to {ARTIFACTS_DIR / 'nlp_probs.parquet'}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Results saved to /home/hrilab/energy-analysis-pipeline/lab_machine_pkg/results/
Best NLP run used for probability export:
model                                          BERT-base
seed                                                   6
accuracy                                        0.445235
f1                                              0.421806
train_time_s                                   20.983316
infer_time_s                                    0.513635
n_test_samples                                      1406
wall_train_start_iso    2026-05-06T21:20:07.546913+00:00
wall_train_end_iso      2026-05-06T21:20:28.530259+00:00
wall_infer_start_iso    2026-05-06T21:20:29.027122+00:00
wall_infer_end_iso      2026-05-06T21:20:29.540762+00:00


Step,Training Loss
500,1.077900


Total NLP experiments: 40
Saved per-headline probabilities to /home/hrilab/energy-analysis-pipeline/lab_machine_pkg/artifacts/nlp_probs.parquet
